<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/08_ctd_distractor_scaling_robust_sft_fast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 - CTD distractor scaling and robust SFT (fast pilot)

Fast Colab/L4 pilot for comparing vanilla LoRA SFT with distractor-aware LoRA SFT.

Speed changes: 2,000 training paths, 100 optimization steps per condition, 100 evaluation examples per condition, batched generation, cached distractor pool, and each adapter is trained exactly once and saved immediately.

In [1]:
!pip -q install -U "transformers>=4.55,<5" "datasets>=3.6,<5" "peft>=0.17,<1" "trl==0.29.1" "accelerate>=1.10,<2" "bitsandbytes>=0.46,<1"


In [2]:
!pip uninstall -y torchao
!pip install -U "torchao>=0.16,<1"

Found existing installation: torchao 0.18.0
Uninstalling torchao-0.18.0:
  Successfully uninstalled torchao-0.18.0
  Using cached torchao-0.18.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (21 kB)
Using cached torchao-0.18.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (3.4 MB)


In [3]:
import os, re, random
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files

if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime in Colab.')
CHEM_GENE='/content/CTD_chem_gene_ixns.tsv.gz'
GENE_DISEASE='/content/CTD_curated_genes_diseases.tsv.gz'
if not os.path.exists(CHEM_GENE) or not os.path.exists(GENE_DISEASE):
    print('Upload CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz')
    files.upload()
assert os.path.exists(CHEM_GENE) and os.path.exists(GENE_DISEASE)
print('GPU:',torch.cuda.get_device_name(0))


GPU: NVIDIA L4


In [4]:
chem_cols=['ChemicalName','ChemicalID','CasRN','GeneSymbol','GeneID','GeneForms','Organism','OrganismID','Interaction','InteractionActions','PubMedIDs']
gd_cols=['GeneSymbol','GeneID','DiseaseName','DiseaseID','DirectEvidence','InferenceChemicalName','InferenceChemicalID','OmimIDs','PubMedIDs']
chem=pd.read_csv(CHEM_GENE,sep='\t',comment='#',header=None,names=chem_cols,dtype=str,low_memory=False)
gd=pd.read_csv(GENE_DISEASE,sep='\t',comment='#',header=None,names=gd_cols,dtype=str,low_memory=False)
chem=chem[chem['OrganismID'].fillna('').str.strip().eq('9606')].copy()
chem=chem[chem['ChemicalName'].notna() & chem['GeneSymbol'].notna() & chem['GeneID'].notna()].copy()
gd=gd[gd['GeneID'].notna() & gd['DiseaseName'].notna() & gd['DiseaseID'].notna()].copy()
chem['GeneID']=chem['GeneID'].str.replace(r'\.0$','',regex=True); gd['GeneID']=gd['GeneID'].str.replace(r'\.0$','',regex=True)
gd=gd.drop_duplicates(['GeneID','DiseaseID'])
pairs=chem.merge(gd[['GeneID','DiseaseName','DiseaseID']],on='GeneID',how='inner')
pairs=pairs.drop_duplicates(['ChemicalID','GeneID','DiseaseID'])[['ChemicalName','ChemicalID','GeneSymbol','GeneID','DiseaseName','DiseaseID']].dropna().reset_index(drop=True)
print('2-hop paths:',len(pairs))


2-hop paths: 4861330


In [5]:
# Fast pilot configuration and chemical-disjoint split.
MAX_TRAIN_EXAMPLES=2000
MAX_EVAL_EXAMPLES=100
DISTRACTOR_COUNTS=[1,3,5,10]
rng=random.Random(42)
chems=pairs['ChemicalID'].drop_duplicates().tolist(); rng.shuffle(chems)
eval_chems=set(chems[:max(1,int(.1*len(chems)))])
train_pool=pairs[~pairs['ChemicalID'].isin(eval_chems)].sample(n=min(MAX_TRAIN_EXAMPLES,len(pairs[~pairs['ChemicalID'].isin(eval_chems)])),random_state=42).reset_index(drop=True)
eval_pool=pairs[pairs['ChemicalID'].isin(eval_chems)].head(MAX_EVAL_EXAMPLES).reset_index(drop=True)
print('Train:',len(train_pool),'Eval:',len(eval_pool))

gene_disease_pool=list({(g,d) for g,d in pairs[['GeneSymbol','DiseaseName']].itertuples(index=False,name=None) if g and d})
print('Cached gene-disease pairs:',len(gene_disease_pool))


Train: 2000 Eval: 100
Cached gene-disease pairs: 33703


In [6]:
def clean_prompt(row):
    return f"Evidence 1: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\nEvidence 2: gene {row.GeneSymbol} is linked to disease {row.DiseaseName}.\nQuestion: What disease is connected to {row.ChemicalName} through gene {row.GeneSymbol}? Return `Disease: <name>` and `Path: Chemical -> Gene -> Disease`."
def clean_answer(row): return f"Disease: {row.DiseaseName}. Path: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}."

def make_distractor_fast(row,k,rng):
    candidates=[x for x in gene_disease_pool if x[0]!=row.GeneSymbol and x[1]!=row.DiseaseName]
    if len(candidates)<k:return None
    ds=rng.sample(candidates,k); edges=[f"{row.GeneSymbol} -> {row.DiseaseName}"]+[f"{g} -> {d}" for g,d in ds]; rng.shuffle(edges)
    p=f"Evidence A: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\nGene-disease evidence:\n- "+'\n- '.join(edges)+f"\nQuestion: Using only the evidence above, what disease is connected to {row.ChemicalName} through gene {row.GeneSymbol}? Return `Disease: <name>` and a short path."
    return p,clean_answer(row)

def make_no_path_fast(row,rng):
    candidates=[x for x in gene_disease_pool if x[0]!=row.GeneSymbol and x[1]!=row.DiseaseName]
    if not candidates:return None
    g,d=rng.choice(candidates)
    p=f"Evidence A: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\nGene-disease evidence: {g} -> {d}.\nQuestion: Is there a supported Chemical -> Gene -> Disease path from {row.ChemicalName} through gene {row.GeneSymbol}? Answer YES or NO."
    return p,'NO. The supplied gene-disease edge does not use the queried gene.'


In [7]:
# Build all evaluation conditions in one pass.
eval_sets={'clean':[],**{f'distractor_{k}':[] for k in DISTRACTOR_COUNTS},'no_path':[]}
for row in eval_pool.itertuples(index=False):
    base={'target_disease':row.DiseaseName,'target_gene':row.GeneSymbol,'target_chemical':row.ChemicalName}
    eval_sets['clean'].append({'prompt':clean_prompt(row),**base,'kind':'clean'})
    for k in DISTRACTOR_COUNTS:
        item=make_distractor_fast(row,k,rng)
        if item: eval_sets[f'distractor_{k}'].append({'prompt':item[0],**base,'kind':f'distractor_{k}'})
    item=make_no_path_fast(row,rng)
    if item: eval_sets['no_path'].append({'prompt':item[0],**base,'target_disease':None,'kind':'no_path'})
for name,items in eval_sets.items():print(f'{name:16s}: {len(items)}')


clean           : 100
distractor_1    : 100
distractor_3    : 100
distractor_5    : 100
distractor_10   : 100
no_path         : 100


In [8]:
from transformers import AutoModelForCausalLM,AutoTokenizer
MODEL_NAME='Qwen/Qwen2.5-0.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
def render(p,a=None):
    m=[{'role':'user','content':p}]+([{'role':'assistant','content':a}] if a else [])
    return tokenizer.apply_chat_template(m,tokenize=False,add_generation_prompt=a is None)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
def make_sft_dataset_fast(df,distractor_prob,seed):
    rr=random.Random(seed); rows=[]
    for row in df.itertuples(index=False):
        if rr.random()<distractor_prob:
            item=make_distractor_fast(row,3,rr); p,a=item if item else (clean_prompt(row),clean_answer(row))
        else:p,a=clean_prompt(row),clean_answer(row)
        rows.append({'text':render(p,a)})
    return Dataset.from_list(rows)
vanilla_ds=make_sft_dataset_fast(train_pool,0.0,1)
robust_ds=make_sft_dataset_fast(train_pool,0.5,2)
print('Vanilla:',len(vanilla_ds),'Robust:',len(robust_ds))


Vanilla: 2000 Robust: 2000


In [13]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)


def train_sft_fast(ds, outdir):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=dtype,
    ).cuda()

    model.config.use_cache = False

    args = SFTConfig(
        output_dir=outdir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        max_steps=100,
        learning_rate=2e-4,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        packing=False,
        gradient_checkpointing=False,
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=ds,
        processing_class=tokenizer,
        peft_config=lora,
    )

    trainer.train()

    # Trainer経由で保存
    trainer.save_model(outdir)
    tokenizer.save_pretrained(outdir)

    return model

In [14]:
print("Training vanilla...")
m = train_sft_fast(
    vanilla_ds,
    "./outputs/08-vanilla-adapter",
)
del m
torch.cuda.empty_cache()

print("Training robust...")
m = train_sft_fast(
    robust_ds,
    "./outputs/08-robust-adapter",
)
del m
torch.cuda.empty_cache()

Training vanilla...


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,2.280400
40,0.979600
60,0.615100
80,0.565700
100,0.566000


In [15]:
def generate_batched(model,prompts,batch_size=16,max_new_tokens=64):
    model.eval(); outs=[]
    for s in range(0,len(prompts),batch_size):
        enc=tokenizer([render(p) for p in prompts[s:s+batch_size]],return_tensors='pt',padding=True,truncation=True,max_length=384)
        enc={k:v.to(model.device) for k,v in enc.items()}
        with torch.no_grad():out=model.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        n=enc['input_ids'].shape[1]; outs.extend(tokenizer.batch_decode(out[:,n:],skip_special_tokens=True))
    return outs
def score_set(items,preds):
    dh=[];ph=[];nh=[]
    for x,pred in zip(items,preds):
        p=re.sub(r'[^a-z0-9]+',' ',pred.lower())
        if x['kind']=='no_path':nh.append('no' in p and 'yes' not in p[:20])
        else:
            d=re.sub(r'[^a-z0-9]+',' ',x['target_disease'].lower());g=re.sub(r'[^a-z0-9]+',' ',x['target_gene'].lower());c=re.sub(r'[^a-z0-9]+',' ',x['target_chemical'].lower())
            dh.append(d in p);ph.append(d in p and g in p and c in p)
    z={}
    if dh:z.update(disease_accuracy=sum(dh)/len(dh),path_accuracy=sum(ph)/len(ph))
    if nh:z['no_path_accuracy']=sum(nh)/len(nh)
    return z


In [16]:
from peft import PeftModel
def evaluate_adapter(path,label):
    base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=dtype).cuda(); m=PeftModel.from_pretrained(base,path); m.eval(); res={}
    for name,items in eval_sets.items():
        preds=generate_batched(m,[x['prompt'] for x in items],batch_size=16,max_new_tokens=64)
        res[name]=score_set(items,preds); print(label,name,res[name])
    del m,base;torch.cuda.empty_cache();return res
vanilla_results=evaluate_adapter('./outputs/08-vanilla-adapter','Vanilla')
robust_results=evaluate_adapter('./outputs/08-robust-adapter','Robust')


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Vanilla clean {'disease_accuracy': 0.99, 'path_accuracy': 0.98}
Vanilla distractor_1 {'disease_accuracy': 0.92, 'path_accuracy': 0.14}
Vanilla distractor_3 {'disease_accuracy': 0.9, 'path_accuracy': 0.15}
Vanilla distractor_5 {'disease_accuracy': 0.87, 'path_accuracy': 0.09}
Vanilla distractor_10 {'disease_accuracy': 0.85, 'path_accuracy': 0.06}
Vanilla no_path {'no_path_accuracy': 0.0}
Robust clean {'disease_accuracy': 1.0, 'path_accuracy': 1.0}
Robust distractor_1 {'disease_accuracy': 0.98, 'path_accuracy': 0.98}
Robust distractor_3 {'disease_accuracy': 1.0, 'path_accuracy': 1.0}
Robust distractor_5 {'disease_accuracy': 0.99, 'path_accuracy': 0.99}
Robust distractor_10 {'disease_accuracy': 0.99, 'path_accuracy': 0.99}
Robust no_path {'no_path_accuracy': 0.0}


In [17]:
print('\nROBUSTNESS COMPARISON')
print('='*72)
print(f"{'Condition':<20}{'Vanilla':>12}{'Robust':>12}{'Delta':>12}")
print('-'*72)
for name in eval_sets:
    metric='no_path_accuracy' if name=='no_path' else 'disease_accuracy'
    v=vanilla_results[name].get(metric,float('nan'));r=robust_results[name].get(metric,float('nan'))
    print(f"{name:<20}{v:>12.3f}{r:>12.3f}{r-v:>12.3f}")



ROBUSTNESS COMPARISON
Condition                Vanilla      Robust       Delta
------------------------------------------------------------------------
clean                      0.990       1.000       0.010
distractor_1               0.920       0.980       0.060
distractor_3               0.900       1.000       0.100
distractor_5               0.870       0.990       0.120
distractor_10              0.850       0.990       0.140
no_path                    0.000       0.000       0.000


## Next step

This is a pilot configuration. If robust SFT shows a clear advantage, rerun the final experiment with larger training/evaluation sets and multiple random seeds. Keep the same chemical-disjoint split logic for comparability.